# FIFA World Cup 2026 Predictor — Project Notebook

**Author initiative** — independent ML forecasting project (walk-forward Poisson / Dixon-Coles).

**Kernel:** Python 3 (ipykernel) → conda env `base` (view-only; no PyTorch needed)

**Before running:** Kernel → **Restart Kernel**, then run cells in order.

Walk-forward backtest:
- **2018** — development (tune iteration)
- **2022** — confirmation (OPTA vs Model vs Actual)
- **2026** — production forecast

Outputs: `outputs/` | Models: `saved_models/` | Report: `Executive_Report.html`

In [1]:
# Cell 1 — Setup (run first after kernel restart)
import sys
from pathlib import Path

# IMPORTANT: import numpy before pandas (avoids copyto errors in Jupyter)
import numpy as np
import pandas as pd

BASE = Path(r"C:\Users\sas_c\OneDrive\Documents\2026\AI Engineer\Course\MS AI Machine Learning\fifa_2026_predictor")
sys.path.insert(0, str(BASE))

from src.config import OUTPUTS, SAVED_MODELS, BACKTEST_CONFIGS, MONTE_CARLO_SIMULATIONS
from src.io_utils import safe_read_csv, resolve_output_file

OUTPUTS.mkdir(exist_ok=True)
SAVED_MODELS.mkdir(exist_ok=True)

# --- Controls ---
SKIP_TRAINING = True    # True = load existing 2018/2022 CSVs. False = re-run backtest.
SKIP_PRODUCTION = True  # True = load existing 2026 CSVs. False = re-run 2026 forecast.
QUICK_RUN = True        # Only used when SKIP_TRAINING or SKIP_PRODUCTION is False
BEST_ITER = 4           # 2018/2022 tables; auto-updated from summary CSV below
ITER_2026 = 5           # 2026 forecast/comparison — your manual choice (not overwritten)
EPOCHS = 50 if QUICK_RUN else 300
SIMULATIONS = 1_000 if QUICK_RUN else MONTE_CARLO_SIMULATIONS
ITERATIONS = (1, 2) if QUICK_RUN else (1, 2, 3, 4)

summary_2018_path = OUTPUTS / "backtest_summary_2018.csv"
if summary_2018_path.exists():
    _s = safe_read_csv(summary_2018_path)
    BEST_ITER = int(_s.loc[_s["brier_score"].idxmin(), "iteration"])

# Resolve 2026 files from ITER_2026 (not BEST_ITER) — keeps manual override
cmp_2026_path, _ = resolve_output_file(OUTPUTS, "comparison_2026_iter{}.csv", ITER_2026)
ITER_2026 = int(cmp_2026_path.stem.split("_iter")[-1])

print(f"numpy {np.__version__} | pandas {pd.__version__}")
print(f"Project:  {BASE}")
print(f"Outputs:  {OUTPUTS}")
print(f"BEST_ITER (2018/2022): {BEST_ITER}  |  ITER_2026: {ITER_2026}")
print(f"SKIP_TRAINING: {SKIP_TRAINING}  |  SKIP_PRODUCTION: {SKIP_PRODUCTION}")

numpy 2.3.5 | pandas 2.3.3
Project:  C:\Users\sas_c\OneDrive\Documents\2026\AI Engineer\Course\MS AI Machine Learning\fifa_2026_predictor
Outputs:  C:\Users\sas_c\OneDrive\Documents\2026\AI Engineer\Course\MS AI Machine Learning\fifa_2026_predictor\outputs
BEST_ITER (2018/2022): 4  |  ITER_2026: 5
SKIP_TRAINING: True  |  SKIP_PRODUCTION: True


## Step 1 — Run backtest (optional)

Set `SKIP_TRAINING = False` in Cell 1 to re-train.  
If you already ran `python run_backtest.py`, keep `SKIP_TRAINING = True` and jump to Step 2.

In [2]:
# Cell 2 — Run walk-forward backtest (skip if outputs already exist)
if SKIP_TRAINING:
    print("Skipping training — loading existing outputs from outputs/")
    if summary_2018_path.exists():
        summary_all = safe_read_csv(OUTPUTS / "backtest_summary_all.csv")
        display(summary_all)
    else:
        print("No backtest_summary_all.csv found. Run: python run_backtest.py")
else:
    import subprocess

    iter_arg = ",".join(str(i) for i in ITERATIONS)
    cmd = [
        "conda", "run", "-n", "deeplearning", "--no-capture-output",
        "python", "run_backtest.py",
        "--iterations", iter_arg,
        "--epochs", str(EPOCHS),
        "--simulations", str(SIMULATIONS),
    ]
    print("Training runs in deeplearning env (not base). Command:")
    print(" ", " ".join(cmd))
    subprocess.run(cmd, cwd=str(BASE), check=True)
    summary_all = safe_read_csv(OUTPUTS / "backtest_summary_all.csv")
    _s = safe_read_csv(summary_2018_path)
    BEST_ITER = int(_s.loc[_s["brier_score"].idxmin(), "iteration"])
    print(f"Updated BEST_ITER = {BEST_ITER}")
    display(summary_all)

Skipping training — loading existing outputs from outputs/


,year,iteration,role,target_date,brier_score,matches_scored,actual_winner,model_top_pick,model_top_pick_pct
0,2018,1,development,2018-06-14,1.203100,64,France,Costa Rica,0.2530
1,2018,2,development,2018-06-14,0.785836,64,France,Brazil,0.1714
2,2018,3,development,2018-06-14,0.759414,64,France,Brazil,0.1839
3,2018,4,development,2018-06-14,0.742398,64,France,Brazil,0.1690
4,2018,5,development,2018-06-14,0.746547,64,France,Brazil,0.1651
5,2022,1,confirmation,2022-11-20,1.091301,64,Argentina,Portugal,0.1414
6,2022,2,confirmation,2022-11-20,0.773673,64,Argentina,Brazil,0.1299
7,2022,3,confirmation,2022-11-20,0.763770,64,Argentina,Brazil,0.2133
8,2022,4,confirmation,2022-11-20,0.736281,64,Argentina,Brazil,0.1309
9,2022,5,confirmation,2022-11-20,0.736155,64,Argentina,Brazil,0.1248


## Step 2 — Brier score comparison (lower is better)

In [3]:
# Cell 3 — Brier summary
summary_2018 = safe_read_csv(OUTPUTS / "backtest_summary_2018.csv")
summary_2022 = safe_read_csv(OUTPUTS / "backtest_summary_2022.csv")

BEST_ITER = int(summary_2018.loc[summary_2018["brier_score"].idxmin(), "iteration"])

print("2018 (development)")
display(summary_2018[["iteration", "brier_score", "model_top_pick", "model_top_pick_pct", "actual_winner"]])

print("\n2022 (confirmation)")
display(summary_2022[["iteration", "brier_score", "model_top_pick", "model_top_pick_pct", "actual_winner"]])

print(f"\nRecommended iteration (lowest 2018 Brier): {BEST_ITER}")

2018 (development)


,iteration,brier_score,model_top_pick,model_top_pick_pct,actual_winner
0,1,1.203100,Costa Rica,0.2530,France
1,2,0.785836,Brazil,0.1714,France
2,3,0.759414,Brazil,0.1839,France
3,4,0.742398,Brazil,0.1690,France
4,5,0.746547,Brazil,0.1651,France



2022 (confirmation)


,iteration,brier_score,model_top_pick,model_top_pick_pct,actual_winner
0,1,1.091301,Portugal,0.1414,Argentina
1,2,0.773673,Brazil,0.1299,Argentina
2,3,0.763770,Brazil,0.2133,Argentina
3,4,0.736281,Brazil,0.1309,Argentina
4,5,0.736155,Brazil,0.1248,Argentina



Recommended iteration (lowest 2018 Brier): 4


## Step 3 — OPTA vs Model vs Actual (tournament win %)

In [4]:
# Cell 4 — Comparison tables (2018 & 2022)
cmp_2018 = safe_read_csv(OUTPUTS / f"comparison_2018_iter{BEST_ITER}.csv")
cmp_2022 = safe_read_csv(OUTPUTS / f"comparison_2022_iter{BEST_ITER}.csv")

def show_comparison(df, year):
    cols = ["opta_rank", "team", "opta_win_pct", "model_win_pct", "actual_winner"]
    view = df[cols].head(15).copy()
    view["opta_win_pct"] = (view["opta_win_pct"] * 100).round(2)
    view["model_win_pct"] = (view["model_win_pct"] * 100).round(2)
    print(f"\n{year} — Top 15 (iteration {BEST_ITER})")
    print(f"Actual winner: {BACKTEST_CONFIGS[year]['actual_winner']}")
    display(view)

show_comparison(cmp_2018, 2018)
show_comparison(cmp_2022, 2022)


2018 — Top 15 (iteration 4)
Actual winner: France


,opta_rank,team,opta_win_pct,model_win_pct,actual_winner
0,1,Brazil,14.20,16.90,False
1,2,Germany,11.40,9.60,False
2,3,Argentina,10.10,5.82,False
3,4,France,9.50,4.76,True
4,5,Spain,8.10,9.81,False
5,6,Belgium,8.20,8.83,False
6,7,Portugal,4.90,4.05,False
7,8,England,1.90,3.11,False
8,9,Uruguay,1.85,4.16,False
9,10,Russia,1.70,0.28,False



2022 — Top 15 (iteration 4)
Actual winner: Argentina


,opta_rank,team,opta_win_pct,model_win_pct,actual_winner
0,1,Brazil,16.30,13.09,False
1,2,Argentina,13.10,7.36,True
2,3,France,12.00,3.31,False
3,4,Spain,8.90,6.71,False
4,5,England,8.80,6.88,False
5,6,Germany,7.70,2.69,False
6,7,Netherlands,6.60,5.31,False
7,8,Portugal,5.80,7.15,False
8,9,Belgium,5.40,4.67,False
9,10,Denmark,3.50,3.97,False


## Step 4 — Match-level backtest sample (2022)

In [5]:
# Cell 5 — Sample match predictions
matches_2022 = safe_read_csv(OUTPUTS / f"backtest_2022_iter{BEST_ITER}.csv")
cols = ["date", "team1", "team2", "team1_score", "team2_score", "p_home", "p_draw", "p_away"]
display(matches_2022[cols].head(10))

,date,team1,team2,team1_score,team2_score,p_home,p_draw,p_away
0,2022-11-20,Qatar,Ecuador,0.0,2.0,0.346775,0.299044,0.354181
1,2022-11-21,Senegal,Netherlands,0.0,2.0,0.500460,0.272739,0.226801
2,2022-11-21,England,Iran,6.0,2.0,0.302487,0.292720,0.404792
3,2022-11-21,USA,Wales,1.0,1.0,0.183140,0.293401,0.523459
4,2022-11-22,Argentina,Saudi Arabia,1.0,2.0,0.182514,0.315144,0.502342
5,2022-11-22,Mexico,Poland,0.0,0.0,0.254872,0.284858,0.460271
6,2022-11-22,Denmark,Tunisia,0.0,0.0,0.285057,0.316048,0.398895
7,2022-11-22,France,Australia,4.0,1.0,0.321200,0.296342,0.382458
8,2022-11-23,Germany,Japan,1.0,2.0,0.435895,0.250635,0.313470
9,2022-11-23,Spain,Costa Rica,7.0,0.0,0.202952,0.313157,0.483891


## Step 5 — 2026 production forecast

Set `SKIP_PRODUCTION = False` to re-train 2026. If `final_2026_probabilities.csv` exists, we load it.

In [6]:
# Cell 6 — 2026 forecast
# Uses SKIP_PRODUCTION from Cell 1 (do not re-import torch here)

win_path = OUTPUTS / f"win_probs_2026_iter{ITER_2026}.csv"
final_path = OUTPUTS / "final_2026_probabilities.csv"
if SKIP_PRODUCTION:
    if win_path.exists():
        print(f"Loading win_probs_2026_iter{ITER_2026}.csv")
        win_2026 = safe_read_csv(win_path)
    elif final_path.exists():
        print(f"Loading final_2026_probabilities.csv (last production run — may not match ITER_2026)")
        win_2026 = safe_read_csv(final_path)
    else:
        raise FileNotFoundError(f"No 2026 probabilities for iter {ITER_2026}. Run: python run_production.py --iteration {ITER_2026}")
else:
    import subprocess

    cmd = [
        "conda", "run", "-n", "deeplearning", "--no-capture-output",
        "python", "run_production.py",
        "--iteration", str(ITER_2026),
        "--epochs", str(EPOCHS),
        "--simulations", str(SIMULATIONS),
    ]
    print("Production runs in deeplearning env (not base). Command:")
    print(" ", " ".join(cmd))
    subprocess.run(cmd, cwd=str(BASE), check=True)
    win_2026 = safe_read_csv(win_path if win_path.exists() else final_path)

win_2026.head(15)

Loading win_probs_2026_iter5.csv


,team,model_win_pct
0,Spain,0.1057
1,Japan,0.0867
2,Argentina,0.0782
3,England,0.0658
4,Brazil,0.0655
5,Morocco,0.0637
6,France,0.0579
7,Portugal,0.0536
8,Germany,0.0406
9,Colombia,0.0361


In [7]:
# Cell 7 — 2026 OPTA vs Model
cmp_2026_path, _ = resolve_output_file(OUTPUTS, "comparison_2026_iter{}.csv", ITER_2026)
cmp_2026 = safe_read_csv(cmp_2026_path)
print(f"Loaded {cmp_2026_path.name}")
view = cmp_2026[["opta_rank", "team", "opta_win_pct", "model_win_pct", "delta_model_minus_opta"]].head(15).copy()
view["opta_win_pct"] = (view["opta_win_pct"] * 100).round(2)
view["model_win_pct"] = (view["model_win_pct"] * 100).round(2)
view["delta_model_minus_opta"] = (view["delta_model_minus_opta"] * 100).round(2)
display(view)

Loaded comparison_2026_iter5.csv


,opta_rank,team,opta_win_pct,model_win_pct,delta_model_minus_opta
0,1,Spain,16.12,10.57,-5.55
1,2,France,12.98,5.79,-7.19
2,3,England,11.18,6.58,-4.60
3,4,Argentina,10.36,7.82,-2.54
4,5,Portugal,7.00,5.36,-1.64
5,6,Brazil,6.61,6.55,-0.06
6,7,Germany,5.12,4.06,-1.06
7,8,Netherlands,3.62,3.42,-0.20
8,9,Norway,3.51,3.19,-0.32
9,10,Belgium,2.37,1.13,-1.24


## Troubleshooting

| Error | Fix |
|-------|-----|
| `numpy._core.multiarray ... copyto` | **Kernel → Restart Kernel**, run Cell 1 first |
| `No module named 'torch'` / `tensor` | Stay on `base` kernel with `SKIP_TRAINING=True` and `SKIP_PRODUCTION=True`. Training uses `deeplearning` via subprocess or terminal |
| `best_iter` / `BEST_ITER` not defined | Run Cell 1 before later cells |
| `comparison_2026_iter1.csv` missing | Normal — 2026 was run at iter 4. Cell 1 sets `ITER_2026` automatically |
| Missing backtest CSVs | `conda activate deeplearning` then `python run_backtest.py` |
| Missing 2026 CSVs | `conda activate deeplearning` then `python run_production.py --iteration 4` |